# Graphql

> Fill in a module description here

In [1]:
# | default_exp graphql

In [2]:
# | hide
from nbdev.showdoc import *  # type: ignore

MVP Spec:
-        save_button = gr.Button("Save Files and Transcriptions")
            save_button.click(
                save_files_and_transcriptions,
                inputs=[user_id_input, user_name_input, file_input],
                outputs=output_text,


        with gr.TabItem("Query and Generate Video"):
            gr.Markdown("## Query and Generate Video")
            gr.Markdown(
                "Enter a query to find relevant utterances and generate a video from them. Use the transcript selector above."
            )
            query_input = gr.Textbox(label="Enter Query")
            select_all_checkbox = gr.Checkbox(label="Select All", value=False)
            user_ids_input_2 = gr.CheckboxGroup(
                label="User IDs",
                choices=get_user_names_and_transcript_counts(db),
                interactive=True,
            )
            select_all_checkbox.change(
                fn=update_user_selection,
                inputs=[select_all_checkbox],
                outputs=[user_ids_input_2],
            )
            fetch_button = gr.Button("Create Video")
            video_download = gr.File(label="Download Video", interactive=False)
            fetch_button.click(
                fetch_utterances,
                inputs=[query_input, user_ids_input_2],
                outputs=video_download,
            )

- Screen 1
    - add user name + metadata +files
    - get back confirmation of file upload

- Screen 2
    - Enter query
    - Get back list of Utterances
    - (frontend only: Select Utterances/words, pick custom times based on word timing) -> UtteranceSegments
    - Input UtteranceSegments, gets back Video with Pending result (+ page redirect)

- Screen 3
    - Video screen - looks at duration and polls more frequently as it gets closer to end.


Graphql spec - claude thinks it should look something like this:

```
type Query {
  getUserNamesAndTranscriptCounts: [UserOption!]!
  getRelevantUtterancesFromQuery(query: String!, transcriptIds: [ID!]!): [Utterance!]!
  getVideo(id: ID!): Video
}

type Mutation {
  saveFilesAndTranscriptions(userId: ID!, userName: String!, files: [Upload!]!): String!
  createVideo(query: String!, utteranceSegments: [UtteranceSegmentInput!]!): Video!
  login(email: String!, password: String!): AuthPayload!
}

type AuthPayload {
  token: String!
  employee: Employee!
}

type Employee {
  id: ID!
  name: String!
  email: String!
  permissionLevel: PermissionLevel!
}

enum PermissionLevel {
  READ
  WRITE
  ADMIN
}

type UserOption {
  label: String!
  value: ID!
}

type Utterance {
  id: ID!
  confidence: Float!
  end: Int!
  speaker: String
  start: Int!
  text: String!
  words: [Word!]!
}

type Word {
  id: ID!
  confidence: Float!
  end: Int!
  speaker: String
  start: Int!
  text: String!
}

input UtteranceSegmentInput {
  utteranceId: ID!
  startWord: ID!
  endWord: ID!
}

type Video {
  id: ID!
  title: String!
  filePath: String
  resolutionX: Int
  resolutionY: Int
  fps: Int
  renderStatus: RenderStatus!
  visibility: VideoVisibility!
  clips: [Clip!]!
}

type Clip {
  id: ID!
  fps: Int!
  resolutionX: Int!
  resolutionY: Int!
  speakerRole: String
  metadataToRender: String
  videoType: VideoType!
  renderStatus: RenderStatus!
  hideMetadata: Boolean!
  words: [Word!]!
}

enum VideoType {
  VIDEO
  AUDIO
}

enum RenderStatus {
  PENDING
  PROCESSING
  COMPLETE
  FAILED
}

enum VideoVisibility {
  PRIVATE
  INTERNAL
  PUBLIC
}
```

In [3]:
# | export
from typing import Annotated, List, Any, Union, cast, Callable
from functools import wraps

from uuid import UUID
import strawberry
import jwt
from jwt.exceptions import InvalidTokenError
from fastapi import FastAPI, Request
from product_horse.db import (
    SqlModelDatabase,
    Employee,
    PermissionLevel as OriginalPermissionLevel,
    Company,
    UnvalidatedCompany,
    UnvalidatedEmployee,
)
from datetime import datetime, timedelta

from strawberry.permission import BasePermission
from strawberry.file_uploads import Upload
from strawberry.fastapi import BaseContext, GraphQLRouter
from dotenv import load_dotenv
import os

load_dotenv()

In [ ]:
# |export
secret = os.getenv("SECRET")

In [ ]:
# | export
database_url = os.getenv("DATABASE_URL")
if database_url is None:
    raise ValueError("DATABASE_URL is not set")
if secret is None:
    raise ValueError("SECRET is not set")
database = SqlModelDatabase(database_url="postgresql://localhost:5432/product_horse")


def create_jwt(employee_id: UUID) -> str:
    expiration = datetime.utcnow() + timedelta(weeks=2)
    payload = {"employee_id": str(employee_id), "exp": expiration}
    return jwt.encode(payload, secret, algorithm="HS256")


def get_employee_id_from_jwt(token: str) -> str:
    payload = jwt.decode(token, secret, algorithms=["HS256"])
    if "employee_id" not in payload:
        raise InvalidTokenError
    if datetime.fromtimestamp(payload["exp"]) < datetime.utcnow():
        raise InvalidTokenError
    return payload["employee_id"]


class Context(BaseContext):
    @property
    def employee(self) -> Employee | None:
        authorization = cast(
            str | None, self.request.headers.get("Authorization", None)
        )
        if authorization is None:
            return None
        if not authorization.startswith("Bearer "):
            return None
        token = authorization.split()[1]
        employee_id = get_employee_id_from_jwt(token)
        return Employee(
            id=employee_id,
            name="Max",
            email="max@producthorse.ai",
            permission_level=OriginalPermissionLevel.ADMIN,
        )


class IsAuthenticated(BasePermission):
    message = "Employee is not authenticated"
    error_extensions = {"code": "UNAUTHORIZED"}

    def has_permission(self, source: Any, info: strawberry.Info, **kwargs) -> bool:
        employee: Employee | None = cast(Employee | None, info.context.employee)
        return employee is not None

In [ ]:
# | export
PermissionsLevel = strawberry.enum(OriginalPermissionLevel)


@strawberry.experimental.pydantic.type(Employee)
class Employee:
    id: UUID
    email: str
    name: str
    permission_level: PermissionsLevel


@strawberry.experimental.pydantic.type(Company)
class Company:
    id: UUID
    name: str
    employees: list[Employee]


@strawberry.type
class RegisterCompanySuccess:
    company: Company


@strawberry.type
class RegisterCompanyError:
    message: str


# Create a Union type to represent the 2 results from the mutation
RegisterResponse = Annotated[
    Union[RegisterCompanySuccess, RegisterCompanyError],
    strawberry.union("RegisterCompanyResponse"),
]


@strawberry.type
class LoginSuccess:
    employee: Employee
    token: str


@strawberry.type
class LoginError:
    message: str


LoginResponse = Annotated[
    Union[LoginSuccess, LoginError],
    strawberry.union("LoginResponse"),
]


@strawberry.type
class Query:
    employee: Employee

    @strawberry.field(permission_classes=[IsAuthenticated])
    def get_user_names_and_transcript_counts(self, info: strawberry.Info) -> str:
        # print('employee', info.context.employee)
        # Dummy resolver
        return


@strawberry.type
class Mutation:
    @strawberry.mutation
    def login(self, email: str, password: str) -> LoginResponse:
        employee = database.authenticate_employee(email, password)
        if employee is None:
            return LoginError(message="Invalid email or password")
        return LoginSuccess(employee=employee, token=create_jwt(employee.id))

    @strawberry.mutation
    def register_company_and_employee(
        self, email: str, password: str, company_name: str
    ) -> RegisterResponse:
        company_to_save = UnvalidatedCompany(name=company_name)
        employee_to_save = UnvalidatedEmployee(email=email, password=password)
        return database.save_company_and_employee(
            email, company_to_save, employee_to_save
        )


async def get_context() -> Context:
    return Context()


schema = strawberry.Schema(Query, Mutation)

graphql_app = GraphQLRouter(
    schema,
    context_getter=get_context,
)

app = FastAPI()
app.include_router(graphql_app, prefix="/graphql")

test queries

```

mutation {
  createVideo(query: "example query", userIds: ["1", "2"]) {
    id
    status
    videoUrl
  }
}

mutation($files: [Upload!]!) {
  saveFilesAndTranscriptions(userId: "123", userName: "John Doe", files: $files)
}

query {
  getRelevantUtterancesFromQuery(query: "example query", transcriptIds: ["1", "2"]) {
    id
    confidence
    end
    speaker
    start
    text
    words {
      id
      confidence
      end
      speaker
      start
      text
    }
  }
}


query {
  getUserNamesAndTranscriptCounts {
    name
    id
    transcriptCount
  }
}
```

In [8]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore